In [1]:
import requests
import pandas as pd
from pathlib import Path
from pystac_client import Client
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Config ─────────────────────────────────────────────────────────────────────
CATALOG_URL = "https://earth-search.aws.element84.com/v1"
OUTPUT_ROOT = Path(r'E:\sentinel_lake_ice\Data\Input\Freezeup S2 Data')
MAX_CLOUD   = 85
YEARS       = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
MAX_WORKERS = 10

# All S2 L2A spectral bands + Scene Classification Layer
BANDS = ["B01", "B02", "B03", "B04", "B05", "B06",
         "B07", "B08", "B8A", "B09", "B11", "B12", "SCL"]

# STAC asset key mapping
ASSET_KEY_MAP = {
    "B01": "coastal",
    "B02": "blue",
    "B03": "green",
    "B04": "red",
    "B05": "rededge1",
    "B06": "rededge2",
    "B07": "rededge3",
    "B08": "nir",
    "B8A": "nir08",
    "B09": "nir09",
    "B11": "swir16",
    "B12": "swir22",
    "SCL": "scl",
}

SITES = {
    'YKD': {
        'aoi_coords': [
            [-162.369230, 60.945820],
            [-162.375829, 61.394512],
            [-161.440110, 61.394512],
            [-161.446709, 60.945820],
            [-162.369230, 60.945820],
        ],
        'freezeup': ('{year}-09-01', '{year}-11-30'),
    },
    'YF': {
        'aoi_coords': [
            [-146.332784, 66.458245],
            [-146.343045, 66.906587],
            [-145.201146, 66.906587],
            [-145.211407, 66.458245],
            [-146.332784, 66.458245],
        ],
        'freezeup': ('{year}-09-01', '{year}-11-30'),
    },
    'NS': {
        'aoi_coords': [
            [-155.902745, 70.351719],
            [-155.917670, 70.799842],
            [-154.555964, 70.799842],
            [-154.570890, 70.351719],
            [-155.902745, 70.351719],
        ],
        'freezeup': ('{year}-09-01', '{year}-11-30'),
    },
}

# ── Helpers ─────────────────────────────────────────────────────────────────────
def make_aoi(coords):
    return {"type": "Polygon", "coordinates": [coords]}


def search_s2_l2a(aoi, start_date, end_date, cloud_cover=85):
    catalog = Client.open(CATALOG_URL)
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        intersects=aoi,
        datetime=f"{start_date}T00:00:00Z/{end_date}T23:59:59Z",
        query={"eo:cloud_cover": {"lt": cloud_cover}},
    )
    return list(search.items())


def download_file(url, out_path):
    """Stream-download a file; skip if already exists."""
    if out_path.exists():
        return
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    with open(out_path, "wb") as f:
        for chunk in r.iter_content(65536):
            f.write(chunk)


def download_scene(item, site_dir):
    """Download all configured bands for one scene."""
    scene_dir = site_dir / item.id
    scene_dir.mkdir(parents=True, exist_ok=True)

    missing = []
    for band in BANDS:
        asset_key = ASSET_KEY_MAP[band]
        if asset_key not in item.assets:
            missing.append(band)
            continue
        download_file(item.assets[asset_key].href, scene_dir / f"{band}.tif")

    if missing:
        print(f"    ! {item.id} - missing assets: {missing}")

    return item, missing


# ── Main loop ───────────────────────────────────────────────────────────────────
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for site_code, cfg in SITES.items():
    print(f"\n{'='*80}")
    print(f"Site: {site_code}")
    print(f"{'='*80}")

    site_dir = OUTPUT_ROOT / site_code
    site_dir.mkdir(parents=True, exist_ok=True)

    aoi = make_aoi(cfg['aoi_coords'])
    start_tpl, end_tpl = cfg['freezeup']
    all_items = []

    # ── Search ────────────────────────────────────────────────────────────────
    for year in YEARS:
        start = start_tpl.format(year=year)
        end   = end_tpl.format(year=year)
        print(f"  Searching {year} ({start} → {end})...")
        items = search_s2_l2a(aoi, start, end, cloud_cover=MAX_CLOUD)
        print(f"    Found {len(items)} scenes")
        all_items.extend(items)

    print(f"\n  Total scenes: {len(all_items)}")
    if not all_items:
        print("  No scenes found — skipping.")
        continue

    # ── Download ──────────────────────────────────────────────────────────────
    print(f"  Downloading {len(BANDS)} bands per scene ({MAX_WORKERS} workers)...")
    metadata = []
    completed = 0

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(download_scene, item, site_dir): item
                   for item in all_items}

        for future in as_completed(futures):
            item, missing = future.result()
            completed += 1
            status = "✗" if missing else "✓"
            print(f"  [{completed:>4}/{len(all_items)}] {status} {item.id}")

            metadata.append({
                'scene_id':      item.id,
                'datetime':      item.properties.get('datetime'),
                'cloud_cover':   item.properties.get('eo:cloud_cover', -1),
                'missing_bands': ','.join(missing) if missing else '',
            })

    # ── Metadata ──────────────────────────────────────────────────────────────
    meta_df   = pd.DataFrame(metadata)
    meta_path = site_dir / 'metadata.csv'
    meta_df.to_csv(meta_path, index=False)

    print(f"\n {site_code} complete")
    print(f"    Scenes : {len(meta_df)}")
    print(f"    Avg cloud cover : {meta_df['cloud_cover'].mean():.1f}%")
    print(f"    Metadata : {meta_path}")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'='*80}")
print("ALL SITES COMPLETE")
print('='*80)

for site_code in SITES:
    meta_path = OUTPUT_ROOT / site_code / 'metadata.csv'
    if meta_path.exists():
        df = pd.read_csv(meta_path)
        incomplete = df[df['missing_bands'] != '']
        print(f"\n  {site_code}:")
        print(f"    Total scenes : {len(df)}")
        print(f"    Avg cloud    : {df['cloud_cover'].mean():.1f}%")
        if not incomplete.empty:
            print(f"    Incomplete   : {len(incomplete)} scenes had missing bands")


Site: YKD
  Searching 2019 (2019-09-01 → 2019-11-30)...
    Found 251 scenes
  Searching 2020 (2020-09-01 → 2020-11-30)...
    Found 226 scenes
  Searching 2021 (2021-09-01 → 2021-11-30)...
    Found 165 scenes
  Searching 2022 (2022-09-01 → 2022-11-30)...
    Found 78 scenes
  Searching 2023 (2023-09-01 → 2023-11-30)...
    Found 89 scenes
  Searching 2024 (2024-09-01 → 2024-11-30)...
    Found 84 scenes
  Searching 2025 (2025-09-01 → 2025-11-30)...
    Found 94 scenes

  Total scenes: 987
  [   1/987] ✓ S2A_3VXJ_20191123_0_L2A
  [   2/987] ✓ S2A_4VCP_20191123_0_L2A
  [   3/987] ✓ S2A_3VXH_20191123_1_L2A
  [   4/987] ✓ S2A_3VXH_20191123_0_L2A
  [   5/987] ✓ S2A_4VCN_20191123_0_L2A
  [   6/987] ✓ S2A_4VCN_20191123_1_L2A
  [   7/987] ✓ S2B_3VXJ_20191121_1_L2A
  [   8/987] ✓ S2B_3VXJ_20191121_0_L2A
  [   9/987] ✓ S2B_4VCP_20191121_1_L2A
  [  10/987] ✓ S2B_3VXH_20191118_1_L2A
  [  11/987] ✓ S2B_4VCP_20191121_0_L2A
  [  12/987] ✓ S2B_3VXH_20191118_0_L2A
  [  13/987] ✓ S2B_3VXJ_20191118_0_